# PPP ensemble analysis

Analysis of the reduced ensembles written by `Executables/PPP/python/ppp_reduce.py` (one file per Markov chain,
`<run>_cNN.h5`):

1. Monte Carlo history
2. autocorrelation times and the bin size
3. thermalization and agreement between chains
4. ergodicity: sign of the fermion determinant
5. single-particle spectrum (electron addition and removal energies, charge gap)
6. equal-time spin and charge correlations
7. two-body correlators: singlet and triplet excitons, pp pairs (stage 2, `ppp_reduce2.py`)

All computations are in `ppp_analysis.py` (tests: `test_ppp_analysis.py`); the Wick contractions are in
`ppp_wick.py` (tested against exact Fock-space traces by `test_ppp_wick.py`). Energies are in units of the
hopping $t$ (and in eV with $t$ from the input file), imaginary time $\tau$ in units of $1/t$. Errors: jackknife
over bins of `BINSIZE` consecutive configurations, with the bins of all chains of a run pooled.

In [ ]:
# --- settings ------------------------------------------------------------------------------------
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "python")))   # Executables/PPP/python

DATA_DIR = os.environ.get("PPP_REDUCED", "~/Research/perylene_b32/reduced")       # the <run>_cNN.h5 files
RUN_ORDER = ["perylene_ohno_b32_Nt128", "perylene_mataga-nishimoto_b32_Nt128", "perylene_hubbard_b32_Nt128"]
LABELS = {"perylene_ohno_b32_Nt128": "Ohno",
          "perylene_mataga-nishimoto_b32_Nt128": "Mataga-Nishimoto",
          "perylene_hubbard_b32_Nt128": "Hubbard"}
DISCARD = 0              # configurations dropped at the start of every chain (a number, or {run: number})
BINSIZE = 10             # configurations per bin; check in section 2
T0, TD = 2, 6            # GEVP: C(TD) v = lambda C(T0) v (time slices)
FIT_WINDOW = (8, 48)     # time slices of the single-exponential fits of the GEVP states, cut for every state
MAX_REL_ERR = 0.25       # where the relative error of its correlator exceeds MAX_REL_ERR
FIT_OVERRIDES = {}       # explicit windows: {(run, class, state): (first, last time slice)}
TAU_PROBE = 8            # time slice of the correlator values used for autocorrelation and chain checks

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import ppp_analysis as pa

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10, "axes.titlesize": 10, "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#8a8a85",
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "lines.linewidth": 1.5, "lines.markersize": 4, "errorbar.capsize": 0,
    "legend.frameon": False, "text.color": "#2b2b29", "axes.labelcolor": "#2b2b29",
    "xtick.color": "#5c5c57", "ytick.color": "#5c5c57"})
pd.set_option("display.precision", 4)

runs = pa.load_runs(DATA_DIR, RUN_ORDER, DISCARD)
assert runs, f"no reduced files in {DATA_DIR}"
label = lambda r: LABELS.get(r, r)
color = {r: pa.COLORS[i] for i, r in enumerate(runs)}

def ppp_block(run):
    return yaml.safe_load(run.yml_text).get("PPP", {})

rows = []
for name, run in runs.items():
    p, c0 = ppp_block(run), run.chains[0]
    rows.append(dict(run=label(name), chains=len(run.chains), configurations=" + ".join(str(c.n) for c in run.chains),
                     Nt=run.Nt, beta=run.beta, delta=run.delta, interaction=p.get("form", "on-site"),
                     U_over_t=run.V[0, 0], t_eV=p.get("hopping energy", np.nan),
                     sources=", ".join(map(str, c0.sources))))
pd.DataFrame(rows).set_index("run")

## 1. Monte Carlo history

The action $S$ and the squared amplitude $a_0^2$ of the time-summed field along the eigenvector of $V$ with the
largest eigenvalue (the uniform charge mode, the slowest mode in earlier runs) for every chain. A drift at the
start of a chain means incomplete thermalization (set `DISCARD`); a chain that sits at a different level from the
others is a slowly decorrelating region (sections 2 and 3).

In [ ]:
fig, axes = plt.subplots(len(runs), 2, figsize=(11, 2.4 * len(runs)), squeeze=False, constrained_layout=True)
for i, (name, run) in enumerate(runs.items()):
    for k, c in enumerate(run.chains):
        x = np.arange(c.n) + c.discard
        axes[i, 0].plot(x, c.S, lw=0.6, color=pa.COLORS[k], label=c.tag[-3:])
        axes[i, 1].plot(x, c.field_modes[:, 0] ** 2, lw=0.6, color=pa.COLORS[k])
    axes[i, 0].set_title(f"{label(name)}: action S")
    axes[i, 1].set_title(f"{label(name)}: uniform charge mode $a_0^2$")
    axes[i, 0].legend(ncol=len(run.chains), loc="upper right", fontsize=8)
for ax in axes[-1]:
    ax.set_xlabel("configuration")
plt.show()

## 2. Autocorrelation times and bin size

$\tau_{\rm int}$ in units of configurations (Gamma method), for the action, its fermionic part, $a_0^2$, and the
addition correlator of the LUMO at time slice `TAU_PROBE` (the removal correlator of the HOMO is identical on every
configuration, see section 5). The plot shows $\tau_{\rm int}$ from
the variance of bin averages as a function of the bin size: it saturates once the bins are independent; rare long
excursions show up there and not in the windowed Gamma method. Bins of `BINSIZE` configurations should be several
times the largest $\tau_{\rm int}$.

In [ ]:
def lumo(run):
    # (class, index in the class block) of the LUMO
    o = int(np.argmin(np.where(run.eps > 0, run.eps, np.inf)))
    cls = run.orbital_class[o]
    return cls, int(np.nonzero(run.classes[cls] == o)[0][0])

def history(c, run):
    return {"S": c.S, "S_fermion": c.S_fermion, "a_0^2": c.field_modes[:, 0] ** 2,
            f"A_LUMO({TAU_PROBE})": c.onebody_element(*lumo(run), TAU_PROBE, "addition")}

histories = {name: [history(c, run) for c in run.chains] for name, run in runs.items()}
rows = []
for name, run in runs.items():
    for c, h in zip(run.chains, histories[name]):
        row = dict(run=label(name), chain=c.tag[-3:], acceptance=np.mean(np.minimum(c.acceptance, 1)))
        for k, x in h.items():
            tau, dtau = pa.tau_int(x)
            row[k] = f"{tau:.2f} ± {dtau:.2f}"
        rows.append(row)
display(pd.DataFrame(rows).set_index(["run", "chain"]))

binsizes = np.array([1, 2, 4, 8, 16, 32, 64])
fig, axes = plt.subplots(1, 2, figsize=(10, 3), constrained_layout=True)
for ax, key, tex in zip(axes, ["S_fermion", "a_0^2"], [r"$S_{\rm fermion}$", r"$a_0^2$"]):
    for name, run in runs.items():
        for k, h in enumerate(histories[name]):
            ok = binsizes <= len(h[key]) // 10
            ax.plot(binsizes[ok], pa.tau_int_binned(h[key], binsizes[ok]), "o-", color=color[name],
                    alpha=1.0 if k == 0 else 0.55, label=label(name) if k == 0 else None)
    ax.axvline(BINSIZE, color="#8a8a85", lw=1, ls=":")
    ax.set_xscale("log", base=2)
    ax.set_xlabel("bin size (configurations)")
    ax.set_title(r"$\tau_{\rm int}$ of " + tex + " from bin averages (one line per chain)")
    ax.set_ylabel(r"$\tau_{\rm int}$ (configurations)")
axes[0].legend()
plt.show()

## 3. Thermalization and agreement between chains

Per chain: the mean (error from bins of `BINSIZE`) and the difference between the first and the second half in
units of its error. Per run: $\chi^2/{\rm dof}$ of the chain means about their weighted mean. Differences of many
standard deviations, or $\chi^2/{\rm dof} \gg 1$, mean incomplete thermalization (first half) or regions of
configuration space that one chain visits and another does not; the spread between chains is then the more
honest error.

In [ ]:
rows, chi = [], []
for name, run in runs.items():
    keys = list(histories[name][0])
    per = {k: [] for k in keys}
    for c, h in zip(run.chains, histories[name]):
        row = dict(run=label(name), chain=c.tag[-3:])
        for k in keys:
            m, e = pa.mean_error(h[k], BINSIZE)
            half = len(h[k]) // 2
            m1, e1 = pa.mean_error(h[k][:half], BINSIZE)
            m2, e2 = pa.mean_error(h[k][half:], BINSIZE)
            row[k] = f"{m:.5g} ± {e:.2g}  ({(m2 - m1) / np.hypot(e1, e2):+.1f}σ)"
            per[k].append((m, e))
        rows.append(row)
    if len(run.chains) > 1:
        chi.append(dict(run=label(name), **{k: f"{pa.consistency(*zip(*v))[2]:.2f}" for k, v in per.items()}))
display(pd.DataFrame(rows).set_index(["run", "chain"]))
print("chi^2/dof of the chain means (second half vs first half in parentheses above):")
display(pd.DataFrame(chi).set_index("run") if chi else "only one chain per run")

## 4. Ergodicity: sign of the fermion determinant

At $\mu = 0$ on a bipartite lattice, $\det M_p\, e^{-i\sum\phi/2}$ is real. Its sign labels regions of
configuration space separated by zeros of the determinant, which the HMC cannot cross; the radial update can.
All chains start at $\phi = 0$, in the $+$ region. If every configuration of every chain has the same sign, this
check cannot distinguish "the other regions do not matter" from "the chains are trapped": compare with a run whose
radial update is effective (radial acceptance around 30 %, arXiv:2504.01575).

In [ ]:
pd.DataFrame([dict(run=label(name), **pa.sign_census(c)) for name, run in runs.items() for c in run.chains]).set_index(["run", "chain"])

## 5. Single-particle spectrum

For each symmetry class, the addition correlator matrix $A_{ab}(\tau) = \langle c_a(\tau) c_b^\dagger(0)\rangle$ of
the class orbitals with $\varepsilon > 0$. The generalized eigenvalue problem $C(\tau_D)\,v = \lambda\, C(\tau_0)\,v$
on the mean gives the states; each jackknife sample is projected on them and fitted with a single exponential. The
fit window of a state is `FIT_WINDOW`, cut where the relative error of its correlator exceeds `MAX_REL_ERR`
(or `FIT_OVERRIDES`). The panels show the effective energies $\log[c(\tau)/c(\tau+1)]/\delta$, the fits (solid,
over the window) and the non-interacting orbital energies $\varepsilon$ (dashed).

Removal energies need no separate analysis: at $\mu = 0$ on a bipartite lattice, particle-hole conjugation gives
$R_{vv'}(\tau) = A_{cc'}(\tau)^*$ on every configuration, for the valence orbitals $v = (-1)^x c$ (tested to $10^{-14}$
in `test_ppp_analysis.py`). The removal spectrum of a class is the addition spectrum of its partner class, and the
charge gap $E_{N+1} + E_{N-1} - 2E_N$ is twice the lowest addition energy.

In [ ]:
spectra = {}
for name, run in runs.items():
    spectra[name] = {}
    for cls in run.classes:
        A, _ = run.onebody(cls, BINSIZE)
        sel = pa.orbital_selection(run, cls, "addition")
        over = {n: w for (r, c, n), w in FIT_OVERRIDES.items() if r == name and c == cls}
        res = pa.state_energies(A, run.delta, T0, TD, FIT_WINDOW, sel, MAX_REL_ERR, over)
        if res is not None:
            res["free"] = np.sort(run.eps[run.classes[cls][sel]])
            spectra[name][cls] = res

for name, run in runs.items():
    classes = list(spectra[name])
    fig, axes = plt.subplots(1, len(classes), figsize=(2.9 * len(classes), 3.0), sharey=True, squeeze=False,
                             constrained_layout=True)
    fig.suptitle(f"{label(name)}: addition energies (effective energies of the GEVP states)", x=0.01, ha="left")
    tmax = 0
    for ax, cls in zip(axes[0], classes):
        res = spectra[name][cls]
        tt = np.arange(res["Eeff"].shape[1]) * run.delta
        for n in range(len(res["E"])):
            good = np.isfinite(res["Eeff"][n]) & (res["Eeff_err"][n] < 0.5)
            last = len(good) if good.all() else int(np.argmin(good))   # signal up to the first bad time slice
            ok = np.arange(len(good)) < last
            ax.errorbar(tt[ok], res["Eeff"][n, ok], res["Eeff_err"][n, ok], fmt="o", ms=2.5, lw=0.8,
                        color=pa.COLORS[n], label=f"state {n}")
            tmax = max(tmax, tt[last - 1] if last else 0)
            if res["windows"][n] is not None:
                ax.plot(np.array(res["windows"][n]) * run.delta, [res["E"][n]] * 2, color=pa.COLORS[n], lw=2.5)
            ax.axhline(res["free"][n], color=pa.COLORS[n], lw=0.8, ls="--")
        ax.set_title(f"class {cls}")
        ax.set_xlabel(r"$\tau\,t$")
    for ax in axes[0]:
        ax.set_xlim(0, 1.15 * tmax + 0.5)
        ax.set_ylim(0, 3.2)
    axes[0, 0].set_ylabel(r"$E_{\rm eff}/t$")
    axes[0, 0].legend(fontsize=7)
    plt.show()

In [ ]:
rows = []
for name, run in runs.items():
    teV = ppp_block(run).get("hopping energy", np.nan)
    for cls, res in spectra[name].items():
        o = run.classes[cls][pa.orbital_selection(run, cls, "addition")[0]]
        partner_cls = run.orbital_class[run.partner[o]] if run.partner[o] >= 0 else "-"
        for n, (E, dE, w, free) in enumerate(zip(res["E"], res["dE"], res["windows"], res["free"])):
            rows.append(dict(run=label(name), cls=cls, state=n, removal_class=partner_cls, E_over_t=E, error=dE,
                             window=f"{w[0] * run.delta:.2f}-{w[1] * run.delta:.2f}" if w else "-",
                             free_eps=free, E_eV=E * teV, error_eV=dE * teV))
pd.DataFrame(rows).set_index(["run", "cls", "state"])

In [ ]:
rows = []
for name, run in runs.items():
    teV = ppp_block(run).get("hopping energy", np.nan)
    cand = [(res["E"][0], res["dE"][0], cls) for cls, res in spectra[name].items() if np.isfinite(res["E"][0])]
    E, dE, cls = min(cand) if cand else (np.nan, np.nan, "-")
    free_gap = np.min(run.eps[run.eps > 0]) - np.max(run.eps[run.eps < 0])
    rows.append(dict(run=label(name), lowest_addition=f"{E:.4f} ± {dE:.4f}  (class {cls})",
                     gap_over_t=f"{2 * E:.4f} ± {2 * dE:.4f}", gap_eV=f"{2 * E * teV:.3f} ± {2 * dE * teV:.3f}",
                     free_gap_over_t=round(free_gap, 4)))
pd.DataFrame(rows).set_index("run")

## 6. Equal-time spin and charge correlations

From $G(t,t)$ of every configuration (Wick's theorem per time slice, averaged over the time slices): total charge
$\langle Q\rangle$ (0 at half filling), total spin $\langle \mathbf S^2\rangle$ (0 for a singlet ground state; at
$\mu = 0$ thermally excited charged states contribute $\sim e^{-\beta E_{\rm add}}$), local moment
$\langle s_x^2\rangle$ and double occupancy $\langle n_{x\uparrow} n_{x\downarrow}\rangle$ (site averages), and the
staggered spin and charge structure factors $S_{\rm AF} = \frac1{N_x}\sum_{xy} e_x e_y \langle s_x s_y\rangle$,
$S_{\rm CDW} = \frac1{N_x}\sum_{xy} e_x e_y \langle q_x q_y\rangle$ ($e_x = \pm1$ the sublattice sign). The plots show
the correlations averaged over site pairs at equal distance.

In [ ]:
eqt, rows = {}, []
for name, run in runs.items():
    G, ss, qq = run.equaltime(BINSIZE)
    obs = pa.equaltime_observables(G, ss, qq, run.sublattice)
    eqt[name] = obs
    me = lambda x: (x.mean(axis=0), x.std(axis=0, ddof=1) / np.sqrt(len(x)))
    row = dict(run=label(name))
    for key, x in (("Q", obs["Q"]), ("S^2", obs["S2"]), ("<s_x^2>", obs["m2"].mean(axis=1)),
                   ("<n_up n_down>", obs["d"].mean(axis=1)), ("S_AF", obs.get("S_AF")), ("S_CDW", obs.get("S_CDW"))):
        if x is not None:
            m, e = me(x)
            row[key] = f"{m:.5f} ± {e:.5f}"
    rows.append(row)
display(pd.DataFrame(rows).set_index("run"))

r, shells = pa.distance_shells(next(iter(runs.values())).positions)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), constrained_layout=True)
for ax, key, title in zip(axes, ("ss", "qq"), (r"spin $\langle s_x s_y\rangle$", r"charge $\langle q_x q_y\rangle$")):
    for name in runs:
        sh = pa.shell_average(eqt[name][key], r, shells)
        ax.errorbar(shells, sh.mean(axis=0), sh.std(axis=0, ddof=1) / np.sqrt(len(sh)), fmt="o-", ms=3,
                    color=color[name], label=label(name))
    ax.axhline(0, color="#8a8a85", lw=0.8)
    ax.set_xlabel("distance (length unit of the input)")
    ax.set_title(title + ", averaged over pairs at equal distance")
axes[0].legend()
plt.show()

## 7. Two-body correlators: excitons and pairs

From stage 2 (`ppp_reduce2.py`): operator bases of orbital pairs from the window of orbitals around the gap, per
symmetry sector (product of the classes of the two orbitals, i.e. the symmetry of the state relative to the ground
state):

* **excitons** $O^\dagger = c^\dagger_c c_v$ (conduction $c$, valence $v$), spin triplet (connected part only) and
  singlet (connected $+\,2\times$ disconnected, minus $2\langle O\rangle\langle O^\dagger\rangle$), in combinations of
  definite particle-hole parity $C$ (spin-symmetric conjugation; $C = (-1)^I$ in the Hubbard model, where $I$ is the
  pseudospin of the notes): $(c,v)$ and $(\bar v, \bar c)$ (partner orbitals) combined with both signs;
* **pp pairs** $c^\dagger_{a\uparrow} c^\dagger_{b\downarrow}$, spin singlet and triplet ($a, b$ conduction). hh pairs are
  the particle-hole images (identical correlators on every configuration).

Energies relative to the $N$-electron ground state, from the GEVP as in section 5. A dipole-allowed (bright) singlet
has $C = -1$ and the symmetry of a position component: sector `-+` ($x$) or `+-` ($y$). Exciton binding energy: charge
gap (section 5) minus the exciton energy. In the Hubbard model, pseudospin symmetry makes the $C=-1$ singlet of the
HOMO-LUMO pair degenerate with the LUMO$^2$ pp pair; the last plot tests this (ratio 1), and shows how far the
long-range interaction moves them apart.

In [ ]:
TWO_BODY_DIR = os.environ.get("PPP_REDUCED2", DATA_DIR.rstrip("/") + "2")   # stage-2 files <run>_cNN.h5
T0_2B, TD_2B = 2, 4          # GEVP time slices for the two-body correlators (t >= 1: t = 0 has contact terms)
FIT_WINDOW_2B = (4, 32)      # cut per state where the relative error exceeds MAX_REL_ERR

two = pa.load_two_body(TWO_BODY_DIR, RUN_ORDER, DISCARD)
blocks = {}
for name, chains in two.items():
    c0, blocks[name] = chains[0], {}
    for (ch, sec), pairs in c0.pairs.items():
        cases = []
        if ch == "exciton":
            for spin in ("singlet", "triplet"):
                M, _ = pa.parity_transform(pairs, c0.partner, spin)
                C = pa.exciton_bins(chains, sec, spin, BINSIZE)
                cases += [((f"exciton {spin}", sec, P), np.einsum("ia,ntab,jb->ntij", Mp, C, Mp)) for P, Mp in M.items() if len(Mp)]
        else:
            cases.append(((ch.replace("_", " "), sec, 0), pa.pp_bins(chains, ch, sec, BINSIZE)))
        for key, C in cases:
            try:
                res = pa.state_energies(C, c0.delta, T0_2B, TD_2B, FIT_WINDOW_2B, None, MAX_REL_ERR)
                res["n_ops"] = C.shape[-1]
                blocks[name][key] = res
            except np.linalg.LinAlgError:
                print(f"{label(name)} {key}: C(T0) not positive definite, skipped")

rows = []
for name in blocks:
    teV = ppp_block(runs[name]).get("hopping energy", np.nan)
    for (ch, sec, P), res in sorted(blocks[name].items()):
        w = res["windows"][0]
        rows.append(dict(run=label(name), channel=ch, sector=sec, C={1: "+1", -1: "-1", 0: ""}[P], operators=res["n_ops"],
                         E0_over_t=res["E"][0], error=res["dE"][0], E0_eV=res["E"][0] * teV,
                         window=f"{w[0] * runs[name].delta:.2f}-{w[1] * runs[name].delta:.2f}" if w else "-"))
pd.DataFrame(rows).set_index(["run", "channel", "sector", "C"])

In [ ]:
def lowest(name, pred):
    cand = [(r["E"][0], r["dE"][0], k) for k, r in blocks[name].items() if pred(k) and np.isfinite(r["E"][0])]
    return min(cand) if cand else (np.nan, np.nan, None)

rows = []
for name in blocks:
    teV = ppp_block(runs[name]).get("hopping energy", np.nan)
    cand = [(r["E"][0], r["dE"][0]) for r in spectra[name].values() if np.isfinite(r["E"][0])]
    Ea, dEa = min(cand) if cand else (np.nan, np.nan)
    gap, dgap = 2 * Ea, 2 * dEa
    S, dS, kS = lowest(name, lambda k: k[0] == "exciton singlet" and k[2] == -1 and k[1] in ("-+", "+-"))
    T, dT, kT = lowest(name, lambda k: k[0] == "exciton triplet")
    P, dP, kP = lowest(name, lambda k: k[0] == "pp singlet")
    f = lambda E, dE: f"{E:.3f} ± {dE:.3f} t = {E * teV:.2f} ± {dE * teV:.2f} eV"     # noqa: E731
    rows.append({"run": label(name), "charge gap": f(gap, dgap),
                 "bright singlet S": f(S, dS) + (f"  ({kS[1]})" if kS else ""),
                 "S binding (gap - S)": f(gap - S, np.hypot(dgap, dS)),
                 "lowest triplet T": f(T, dT) + (f"  ({kT[1]}, C={kT[2]:+d})" if kT else ""),
                 "S - T": f(S - T, np.hypot(dS, dT)),
                 "pp singlet - 2 E_add": f(P - 2 * Ea, np.hypot(dP, 2 * dEa))})
display(pd.DataFrame(rows).set_index("run").T)
print("errors combined in quadrature (separate fits on the same bins); a joint jackknife follows when needed")

In [ ]:
keys = [("exciton singlet", "-+", -1, "bright singlet (sector -+, C=-1)"),
        ("exciton triplet", "-+", +1, "triplet (sector -+, C=+1)"),
        ("pp singlet", "++", 0, "pp singlet (sector ++)")]
for name in blocks:
    run = runs[name]
    fig, axes = plt.subplots(1, len(keys), figsize=(3.6 * len(keys), 3.0), sharey=True, constrained_layout=True)
    fig.suptitle(f"{label(name)}: two-body effective energies (GEVP states)", x=0.01, ha="left")
    for ax, (ch, sec, P, title) in zip(axes, keys):
        res = blocks[name].get((ch, sec, P))
        ax.set_title(title)
        ax.set_xlabel(r"$\tau\,t$")
        if res is None:
            continue
        tt = np.arange(res["Eeff"].shape[1]) * run.delta
        tmax = 0
        for n in range(min(3, len(res["E"]))):
            good = np.isfinite(res["Eeff"][n]) & (res["Eeff_err"][n] < 0.5)
            good[0] = False                                   # t = 0: contact terms
            good[run.Nt // 2:] = False                        # beyond beta/2: backward propagation
            first = int(np.argmax(good)) if good.any() else 0
            last = first + (int(np.argmin(good[first:])) if not good[first:].all() else len(good) - first)
            ok = (np.arange(len(good)) >= first) & (np.arange(len(good)) < last)
            ax.errorbar(tt[ok], res["Eeff"][n, ok], res["Eeff_err"][n, ok], fmt="o", ms=2.5, lw=0.8,
                        color=pa.COLORS[n], label=f"state {n}")
            if res["windows"][n] is not None:
                ax.plot(np.array(res["windows"][n]) * run.delta, [res["E"][n]] * 2, color=pa.COLORS[n], lw=2.5)
            if ok.any():
                tmax = max(tmax, tt[ok].max())
        ax.set_xlim(0, min(1.15 * tmax + 0.5, run.Nt * run.delta / 2))     # beyond beta/2: backward propagation
        ax.set_ylim(0, 5)
    axes[0].set_ylabel(r"$E_{\rm eff}/t$")
    axes[0].legend(fontsize=7)
    plt.show()

In [ ]:
# pseudospin test: C = -1 singlet of the HOMO-LUMO pair over the LUMO^2 pp singlet (1 in the Hubbard model)
fig, ax = plt.subplots(figsize=(6, 3.2), constrained_layout=True)
for name, chains in two.items():
    c0 = chains[0]
    lumo = int(np.argmin(np.where(c0.eps > 0, c0.eps, np.inf)))
    homo = int(c0.partner[lumo])
    sec_e = next(s for (ch, s), p in c0.pairs.items() if ch == "exciton" and any((a, b) == (lumo, homo) for a, b in p))
    sec_p = next(s for (ch, s), p in c0.pairs.items() if ch == "pp_singlet" and any((a, b) == (lumo, lumo) for a, b in p))
    ie = [i for i, (a, b) in enumerate(c0.pairs[("exciton", sec_e)]) if (a, b) == (lumo, homo)][0]
    ip = [i for i, (a, b) in enumerate(c0.pairs[("pp_singlet", sec_p)]) if (a, b) == (lumo, lumo)][0]
    Ce = pa.exciton_bins(chains, sec_e, "singlet", BINSIZE)[:, :, ie, ie]
    Cp = pa.pp_bins(chains, "pp_singlet", sec_p, BINSIZE)[:, :, ip, ip]
    ratio, dratio, _ = pa.jk_apply(lambda m: m[:, 0] / m[:, 1], np.stack([Ce, Cp], axis=-1))
    tt = np.arange(len(ratio)) * c0.delta
    good = np.isfinite(ratio) & (dratio < 0.5 * np.abs(ratio))
    good[0], good[len(ratio) // 2:] = False, False             # t = 0: contact terms; beyond beta/2: backward
    last = 1 + (int(np.argmin(good[1:])) if not good[1:].all() else len(good) - 1)
    ok = (np.arange(len(ratio)) >= 1) & (np.arange(len(ratio)) < last)   # contiguous signal from t = 1
    ax.errorbar(tt[ok], ratio[ok], dratio[ok], fmt="o-", ms=3, lw=1, color=color[name], label=label(name))
ax.axhline(1, color="#8a8a85", lw=0.8, ls="--")
ax.set_xlabel(r"$\tau\,t$")
ax.set_title("HOMO-LUMO singlet exciton / LUMO$^2$ pp pair (pseudospin partners)")
ax.legend()
plt.show()